In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
FEATURE_PATH = "cohort_feature_matrix.csv"

df_feat = pd.read_csv(FEATURE_PATH)
print("Loaded:", df_feat.shape)
print("Columns (first 25):", df_feat.columns[:25].tolist())

# --- Parse timestamps ---
for c in ["intime", "outtime", "charttime_hour"]:
    if c in df_feat.columns:
        df_feat[c] = pd.to_datetime(df_feat[c], errors="coerce")

# --- Create an integer hour index like your old code expects ---
# old code expects: KEYS = ["subject_id","hadm_id","stay_id","hour"]
if "hour" not in df_feat.columns:
    if {"charttime_hour", "intime"}.issubset(df_feat.columns):
        df_feat["hour"] = ((df_feat["charttime_hour"] - df_feat["intime"]).dt.total_seconds() / 3600.0)
        df_feat["hour"] = np.floor(df_feat["hour"]).astype("Int64")
    else:
        raise ValueError("Need either an existing 'hour' column or both 'charttime_hour' and 'intime' to compute hour.")

# Basic cleanup
df_feat = df_feat.dropna(subset=["stay_id", "hour"]).copy()
df_feat["hour"] = df_feat["hour"].astype(int)

KEYS = ["subject_id", "hadm_id", "stay_id", "hour"]
print("Sanity:", df_feat[KEYS].head())


In [ ]:
LABEL_PATH = "sepsis_comorbidity.csv"   # change if your label file has a different name/path

def sepsis_to_01(x: pd.Series) -> pd.Series:
    s = x.copy()
    if pd.api.types.is_bool_dtype(s):
        return s.astype(int)
    ss = s.astype(str).str.strip().str.lower()
    mapping = {"true":1, "false":0, "1":1, "0":0, "yes":1, "no":0, "t":1, "f":0}
    out = ss.map(mapping)
    out_num = pd.to_numeric(s, errors="coerce")
    out = out.fillna(out_num)
    return out.fillna(0).astype(int)

# Try to load labels (if the file exists in your working dir)
try:
    df_lab = pd.read_csv(LABEL_PATH)
    print("Loaded label file:", df_lab.shape)
except FileNotFoundError:
    df_lab = None
    print(f"⚠️ Could not find {LABEL_PATH}. If you want the same y_next_1to4h label, you must provide/point to this file.")

if df_lab is None:
    raise ValueError(
        "No label file found. You need a file containing at least: "
        "['subject_id','hadm_id','stay_id','sepsis3'] and one onset-time column "
        "('sofa_time' OR 'suspected_infection_time') to reproduce y_next_1to4h."
    )

# Parse time columns if present
for c in ["sofa_time","suspected_infection_time","admittime","dischtime","antibiotic_time","culture_time"]:
    if c in df_lab.columns:
        df_lab[c] = pd.to_datetime(df_lab[c], errors="coerce")

# Merge stay-level labels onto hourly feature matrix
data = df_feat.merge(df_lab, on=["subject_id","hadm_id","stay_id"], how="left")

# sepsis indicator
if "sepsis3" not in data.columns:
    raise ValueError("Label file must contain 'sepsis3' to match your old notebook logic.")
data["sepsis3_bin"] = sepsis_to_01(data["sepsis3"])

# choose onset column
if "sofa_time" in data.columns and data["sofa_time"].notna().any():
    onset_col = "sofa_time"
elif "suspected_infection_time" in data.columns and data["suspected_infection_time"].notna().any():
    onset_col = "suspected_infection_time"
else:
    raise ValueError("Need non-null 'sofa_time' or 'suspected_infection_time' to label onset in next 1–4h.")

data["onset_time"] = pd.to_datetime(data[onset_col], errors="coerce")

# current time at each hour (same as your old code)
if "intime" not in data.columns or data["intime"].isna().all():
    raise ValueError("Need 'intime' in feature matrix to compute t0 (hour timestamp).")

data["t0"] = data["intime"] + pd.to_timedelta(data["hour"], unit="h")

# label: onset in (t0, t0+4h]
data["y_next_1to4h"] = (
    (data["sepsis3_bin"] == 1) &
    (data["onset_time"].notna()) &
    (data["onset_time"] > data["t0"]) &
    (data["onset_time"] <= data["t0"] + pd.Timedelta(hours=4))
).astype(int)

# OPTIONAL: drop rows at/after onset
data_labeled = data[(data["onset_time"].isna()) | (data["t0"] < data["onset_time"])].copy()

print("Using onset column:", onset_col)
print("Label distribution:", data_labeled["y_next_1to4h"].value_counts(dropna=False).to_dict())

stay_label = data_labeled.groupby("stay_id")["y_next_1to4h"].max()
pos_stays = set(stay_label[stay_label == 1].index)
neg_stays = set(stay_label[stay_label == 0].index)
print("Num positive stays:", len(pos_stays))
print("Num negative stays:", len(neg_stays))

if len(pos_stays) == 0:
    raise ValueError(
        "Still 0 positive stays after labeling. "
        "Check that onset_time is within the ICU window and hour aligns to intime."
    )


In [ ]:
def split_stays_with_pos(pos_stays, neg_stays, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    pos_stays = np.array(list(pos_stays))
    neg_stays = np.array(list(neg_stays))
    rng.shuffle(pos_stays); rng.shuffle(neg_stays)

    if len(pos_stays) == 0:
        return set(neg_stays), set()

    n_pos_test = max(1, int(len(pos_stays) * test_size))
    n_neg_test = int(len(neg_stays) * test_size)

    test_stays = np.concatenate([pos_stays[:n_pos_test], neg_stays[:n_neg_test]])
    train_stays = np.concatenate([pos_stays[n_pos_test:], neg_stays[n_neg_test:]])
    return set(train_stays), set(test_stays)

drop_cols = set(KEYS + [
    "sepsis3","sepsis3_bin","y_next_1to4h",
    "onset_time","t0",
    # time columns that can leak / be non-numeric
    "charttime_hour","outtime",
    "suspected_infection_time","sofa_time",
    "admittime","dischtime","antibiotic_time","culture_time"
])
feature_cols = [c for c in data_labeled.columns if c not in drop_cols]

X = data_labeled[feature_cols]
y = data_labeled["y_next_1to4h"].astype(int)

train_stays, test_stays = split_stays_with_pos(pos_stays, neg_stays, test_size=0.2, seed=42)
train_mask = data_labeled["stay_id"].isin(train_stays)
test_mask  = data_labeled["stay_id"].isin(test_stays)

X_train, X_test = X.loc[train_mask].copy(), X.loc[test_mask].copy()
y_train, y_test = y.loc[train_mask].copy(), y.loc[test_mask].copy()

print("y_train:", y_train.value_counts().to_dict(), " | y_test:", y_test.value_counts().to_dict())




# --- FIX: remove datetime columns from X before preprocessing ---
dt_cols = X_train.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()
if len(dt_cols) > 0:
    print("Dropping datetime columns from features:", dt_cols)
    X_train = X_train.drop(columns=dt_cols)
    X_test  = X_test.drop(columns=dt_cols)

# Recompute cat/num after dropping datetimes
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ],
    remainder="drop"
)

model = Pipeline([
    ("prep", preprocess),
    ("rf", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    ))
])

model.fit(X_train, y_train)


In [ ]:
proba = model.predict_proba(X_test)
if proba.shape[1] < 2:
    raise RuntimeError("Model trained on one class unexpectedly. Check y_train distribution above.")
proba_test = proba[:, 1]

if y_test.nunique() == 2:
    print("ROC-AUC:", roc_auc_score(y_test, proba_test))
    print("PR-AUC:", average_precision_score(y_test, proba_test))
else:
    print("⚠️ y_test has one class; AUC metrics not defined. y_test:", y_test.value_counts().to_dict())

data_labeled.loc[X_test.index, "pred_proba_next_1to4h"] = proba_test

display_cols = ["stay_id","hour","y_next_1to4h","pred_proba_next_1to4h"]
if "sofa_time" in data_labeled.columns: display_cols.append("sofa_time")
elif "suspected_infection_time" in data_labeled.columns: display_cols.append("suspected_infection_time")

print(data_labeled[display_cols].head(30))


In [ ]:
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, accuracy_score,
    balanced_accuracy_score, roc_auc_score, average_precision_score
)

y_true = data_labeled.loc[X_test.index, "y_next_1to4h"].astype(int).values
p_hat  = data_labeled.loc[X_test.index, "pred_proba_next_1to4h"].astype(float).values

def eval_at_threshold(y_true, p_hat, thr=0.5):
    y_pred = (p_hat >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    precision   = tp / (tp + fp) if (tp + fp) else np.nan
    npv         = tn / (tn + fn) if (tn + fn) else np.nan

    acc   = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else np.nan
    bacc  = (sensitivity + specificity) / 2 if np.isfinite(sensitivity) and np.isfinite(specificity) else np.nan
    f1    = (2 * precision * sensitivity / (precision + sensitivity)) if (precision + sensitivity) else np.nan

    out = {
        "thr": thr,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "sensitivity(recall)": sensitivity,
        "specificity": specificity,
        "precision(PPV)": precision,
        "NPV": npv,
        "accuracy": acc,
        "balanced_accuracy": bacc,
        "f1": f1
    }
    return out

# single threshold
print(eval_at_threshold(y_true, p_hat, thr=0.5))

# sweep thresholds
ths = np.linspace(0.05, 0.95, 19)
rows = [eval_at_threshold(y_true, p_hat, thr=t) for t in ths]
df_thr = pd.DataFrame(rows).sort_values("f1", ascending=False)
print(df_thr.head(10))


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score

# ------------------------------------------------------------
# 0) Ensure we have predicted probabilities aligned to X_test
# ------------------------------------------------------------
# If your pipeline is called `model` and X_test is a DataFrame:
p_hat = model.predict_proba(X_test)[:, 1]

# Ground-truth labels aligned to X_test
# (works whether you kept y_test as a Series or want to pull from data_labeled)
if "y_test" in globals():
    y_true = np.array(y_test).astype(int)
else:
    y_true = data_labeled.loc[X_test.index, "y_next_1to4h"].astype(int).values

# Store in data_labeled for easy inspection later (optional but useful)
data_labeled.loc[X_test.index, "pred_proba_next_1to4h"] = p_hat

# ------------------------------------------------------------
# 1) Metric function at a threshold
# ------------------------------------------------------------
def eval_at_threshold(y_true, p_hat, thr):
    y_pred = (p_hat >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan   # recall / TPR
    specificity = tn / (tn + fp) if (tn + fp) else np.nan   # TNR
    precision   = tp / (tp + fp) if (tp + fp) else np.nan   # PPV
    npv         = tn / (tn + fn) if (tn + fn) else np.nan   # NPV

    acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else np.nan
    bal_acc = (sensitivity + specificity) / 2 if (np.isfinite(sensitivity) and np.isfinite(specificity)) else np.nan
    f1 = (2 * precision * sensitivity / (precision + sensitivity)) if (precision + sensitivity) else np.nan

    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    fnr = fn / (fn + tp) if (fn + tp) else np.nan

    return {
        "threshold": thr,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "accuracy": acc,
        "balanced_acc": bal_acc,
        "precision": precision,
        "recall/sensitivity": sensitivity,
        "specificity": specificity,
        "NPV": npv,
        "F1": f1,
        "FPR": fpr,
        "FNR": fnr
    }

# ------------------------------------------------------------
# 2) Threshold-free summary (AUC metrics)
# ------------------------------------------------------------
summary = {}
if len(np.unique(y_true)) == 2:
    summary["ROC-AUC"] = roc_auc_score(y_true, p_hat)
    summary["PR-AUC"]  = average_precision_score(y_true, p_hat)
else:
    summary["ROC-AUC"] = np.nan
    summary["PR-AUC"]  = np.nan

summary_df = pd.DataFrame([summary])
print("=== Threshold-free summary (test set) ===")
display(summary_df.style.format("{:.4f}"))

# ------------------------------------------------------------
# 3) Metrics at selected thresholds
# ------------------------------------------------------------
threshold_list = [0.1, 0.2, 0.3, 0.5]
rows = [eval_at_threshold(y_true, p_hat, t) for t in threshold_list]
metrics_df = pd.DataFrame(rows)

print("=== Metrics at selected thresholds (test set) ===")
display(
    metrics_df[[
        "threshold","TP","FP","TN","FN",
        "precision","recall/sensitivity","specificity","F1",
        "accuracy","balanced_acc","NPV","FPR","FNR"
    ]].style
      .format({
          "precision":"{:.3f}",
          "recall/sensitivity":"{:.3f}",
          "specificity":"{:.3f}",
          "F1":"{:.3f}",
          "accuracy":"{:.3f}",
          "balanced_acc":"{:.3f}",
          "NPV":"{:.3f}",
          "FPR":"{:.3f}",
          "FNR":"{:.3f}",
      })
)

# ------------------------------------------------------------
# 4) Threshold sweep + best operating points
# ------------------------------------------------------------
thresholds = np.linspace(0.01, 0.99, 99)
sweep = pd.DataFrame([eval_at_threshold(y_true, p_hat, t) for t in thresholds])

# Best F1 threshold
best_f1 = sweep.iloc[sweep["F1"].idxmax()][[
    "threshold","F1","precision","recall/sensitivity","specificity","TP","FP","TN","FN"
]]

# Best specificity with recall >= target
target_recall = 0.80
eligible = sweep[sweep["recall/sensitivity"] >= target_recall]
best_spec = None
if len(eligible) > 0:
    best_spec = eligible.iloc[eligible["specificity"].idxmax()][[
        "threshold","specificity","recall/sensitivity","precision","F1","TP","FP","TN","FN"
    ]]

print("=== Best operating points (test set) ===")
best_tbl = pd.DataFrame([best_f1])
best_tbl["criterion"] = "Best F1"
best_tbl = best_tbl[["criterion"] + [c for c in best_tbl.columns if c != "criterion"]]

if best_spec is not None:
    tmp = pd.DataFrame([best_spec])
    tmp["criterion"] = f"Max specificity with recall ≥ {target_recall}"
    tmp = tmp[["criterion"] + [c for c in tmp.columns if c != "criterion"]]
    best_tbl = pd.concat([best_tbl, tmp], ignore_index=True)

display(
    best_tbl.style.format({
        "threshold":"{:.2f}",
        "F1":"{:.3f}",
        "precision":"{:.3f}",
        "recall/sensitivity":"{:.3f}",
        "specificity":"{:.3f}",
    })
)

# Optional: show top 10 by F1
print("=== Top 10 thresholds by F1 ===")
display(
    sweep.sort_values("F1", ascending=False).head(10)[
        ["threshold","F1","precision","recall/sensitivity","specificity","TP","FP","TN","FN"]
    ].style.format({
        "threshold":"{:.2f}",
        "F1":"{:.3f}",
        "precision":"{:.3f}",
        "recall/sensitivity":"{:.3f}",
        "specificity":"{:.3f}",
    })
)
